# 05 - Advanced Experiments and Feature Engineering

This notebook is for controlled experimentation beyond the core survival model:
1. Interaction terms and duplicate-signal cleanup
2. Stability transforms for skewed features
3. Horizon-specific binary models
4. Calibration studies for experimental comparisons
5. Ensembling and optional AutoML benchmark checks

## Experiment Rules
- Keep one change per experiment.
- Use fixed splits and seeds.
- Record C-index, weighted Brier, and calibration diagnostics.

## Feature Engineering Rationale

The EDA and correlation plots suggest that `dist_min_ci_0_5h` is the strongest single signal, while several other columns are either highly correlated or represent similar physical quantities in different forms. That means the model can easily over-focus on distance and underuse the contextual information in speed, direction, and growth.

The feature-engineering step below serves two purposes:
1. Remove duplicate signals that do not add new information.
2. Create interaction features that describe threat in context, especially when two fires are at similar distances but behave differently.

This is useful because the model should not only learn whether a fire is close, but also how quickly it is approaching, whether it is aligned toward a zone, and whether growth is reinforcing that motion.

## Skew Handling and Scale Stability

The EDA typically shows that many physical wildfire features are strongly right-skewed. A few very large fires can dominate the raw scale, which makes some models overreact to extreme values and ignore the middle of the distribution.

To reduce that effect, the notebook keeps two reusable preprocessing paths:
1. `log1p_...` transforms for skewed positive variables.
2. `RobustScaler` for experiments that need median-based scaling instead of standard deviation-based scaling.

This does not replace the raw features. It creates a second, more stable representation that can be tested in later modeling notebooks when comparing calibration and ranking performance.

In [19]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import RobustScaler

DATASET_DIR = Path('../dataset')
train = pd.read_csv(DATASET_DIR / 'train.csv').fillna(0)


def add_feature_engineering(df):
    df = df.copy()

    # Drop exact or near-exact duplicates so the model does not waste capacity
    # on multiple columns that encode the same physical signal.
    df = df.drop(columns=['relative_growth_0_5h', 'closing_speed_abs_m_per_h'], errors='ignore')

    # Interaction features help the model reason about threat in context.
    # Distance alone is strong, but it does not describe how quickly the fire
    # is approaching, whether it is aligned toward a zone, or how growth and
    # direction combine when distance is already small.
    df['time_to_contact'] = df['dist_min_ci_0_5h'] / (df['closing_speed_m_per_h'].abs() + 1.0)
    df['threat_index'] = (df['closing_speed_m_per_h'].abs() * df['alignment_abs']) / (df['dist_min_ci_0_5h'] + 1.0)
    df['growth_pressure'] = df['area_growth_rate_ha_per_h'] * df['alignment_abs']
    df['distance_growth_balance'] = df['dist_min_ci_0_5h'] / (df['area_growth_rate_ha_per_h'].abs() + 1.0)
    df['directional_urgency'] = df['along_track_speed'] * df['alignment_abs']

    return df.fillna(0)


def add_stability_transforms(df):
    df = df.copy()

    # Several columns are highly skewed. Log transforms and robust scaling are
    # useful for later model variants that benefit from more stable numeric ranges.
    skewed_positive_cols = [
        'area_first_ha',
        'area_growth_abs_0_5h',
        'area_growth_rate_ha_per_h',
        'radial_growth_m',
        'radial_growth_rate_m_per_h',
        'dist_min_ci_0_5h',
        'dist_std_ci_0_5h',
        'dist_change_ci_0_5h',
        'projected_advance_m',
        'closing_speed_m_per_h',
        'centroid_displacement_m',
        'centroid_speed_m_per_h',
    ]

    for col in skewed_positive_cols:
        if col in df.columns:
            df[f'log1p_{col}'] = np.log1p(np.clip(df[col], a_min=0, a_max=None))

    return df


def build_model_frame(df, use_stability_transforms=True):
    df = add_feature_engineering(df)
    if use_stability_transforms:
        df = add_stability_transforms(df)
    return df


train_fe = build_model_frame(train, use_stability_transforms=True)
features = [c for c in train_fe.columns if c not in ['event_id', 'time_to_hit_hours', 'event']]
X = train_fe[features]

# A robust-scaled version is kept here for later experiments that need it.
robust_scaler = RobustScaler()
X_robust = pd.DataFrame(robust_scaler.fit_transform(X), columns=features, index=X.index)

## Horizon Framing Utility
Create binary labels by horizon for optional calibration and classification-style benchmarking.

In [20]:
def horizon_label(df, horizon):
    known_mask = (df['event'] == 1) | (df['time_to_hit_hours'] >= horizon)
    y = ((df['event'] == 1) & (df['time_to_hit_hours'] <= horizon)).astype(int)
    return y, known_mask

for h in [12, 24, 48, 72]:
    y_h, m_h = horizon_label(train, h)
    print(f'h={h}: usable={m_h.sum()} positives={y_h[m_h].sum()}')

h=12: usable=215 positives=49
h=24: usable=196 positives=63
h=48: usable=166 positives=66
h=72: usable=69 positives=69


## Optional: AutoGluon Benchmark
AutoML can be used for horizon-specific probability benchmarking.
For strict censoring-aware modeling, survival-focused methods remain primary.

In [21]:
# Optional install:
# %pip install autogluon

print('AutoML benchmark cell placeholder ready.')

AutoML benchmark cell placeholder ready.
